In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)
from io import StringIO

import logging
logger = logging.getLogger("test_session_helpers")
logger.addHandler(logging.NullHandler())


In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- session_dropna_unique ---
FIX_SESSION_DROPNA_UNIQUE_DATABASE_INFO = {"db": pd.Series(["val1", "val2", None, "val1"]), "host": "localhost", "port": 5432, "dbname": "test"}
FIX_SESSION_DROPNA_UNIQUE_DATABASE_INFO_PL = {"db": pl.Series("db", ["val1", "val2", None, "val1"]), "host": "localhost", "port": 5432, "dbname": "test"}
FIX_SESSION_DROPNA_UNIQUE_EMPTY_INFO = {"db": pd.Series([], dtype="object"), "host": "localhost", "port": 5432, "dbname": "test"}
FIX_SESSION_DROPNA_UNIQUE_EMPTY_INFO_PL = {"db": pl.Series("db", [], dtype=pl.Utf8), "host": "localhost", "port": 5432, "dbname": "test"}

# --- session_read_csv ---
FIX_SESSION_READ_CSV_QUERY_RESULT = "db\nval1\nval2\nval3"  # CSV string for pd.read_csv(StringIO(...))
FIX_SESSION_READ_CSV_EMPTY_QUERY_RESULT = "db\n"

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_session_dropna_unique(database_info):
    unique_values = database_info["db"].dropna().unique()
    return unique_values

def before_session_read_csv(query_result):
    database_info = pd.read_csv(StringIO(query_result))

    if database_info.empty or "db" not in database_info.columns:
        logger.warning("No column information found in the RDF store")
        return None

    unique_values = database_info["db"].dropna().unique()
    return unique_values

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_session_dropna_unique(database_info):

    unique_values = database_info["db"].drop_nulls()
    if unique_values.dtype in (pl.Float32, pl.Float64):
        unique_values = unique_values.drop_nans()
    unique_values = unique_values.unique(maintain_order=True)
    return unique_values

def gen_session_read_csv(query_result):
    from io import StringIO

    database_info = pl.read_csv(StringIO(query_result))

    if database_info.is_empty() or "db" not in database_info.columns:
        logger.warning("No column information found in the RDF store")
        return None

    unique_values = database_info["db"].drop_nulls().unique(maintain_order=True)
    return unique_values

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pl.Series): return pl.DataFrame({"value": r.to_list()})
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.DataFrame({"value": r.tolist()})
    if isinstance(r, np.ndarray): return pl.DataFrame({"value": r.tolist()})
    if isinstance(r, (list, tuple)): return pl.DataFrame({"value": list(r)})
    if hasattr(r, "to_list") and callable(r.to_list): return pl.DataFrame({"value": r.to_list()})
    if hasattr(r, "tolist") and callable(r.tolist): return pl.DataFrame({"value": r.tolist()})
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: session_dropna_unique ===

# L1 smoke – generated
try:
    _r = gen_session_dropna_unique(FIX_SESSION_DROPNA_UNIQUE_DATABASE_INFO_PL)
    print("✅ L1 smoke gen_session_dropna_unique: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_session_dropna_unique: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_session_dropna_unique(FIX_SESSION_DROPNA_UNIQUE_DATABASE_INFO)
    print("✅ L1 smoke before_session_dropna_unique: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_session_dropna_unique: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_session_dropna_unique(FIX_SESSION_DROPNA_UNIQUE_DATABASE_INFO)
    _rg = gen_session_dropna_unique(FIX_SESSION_DROPNA_UNIQUE_DATABASE_INFO_PL)
    compare(_rb, _rg, "session_dropna_unique")
except Exception as _e:
    print(f"❌ L2 equivalence session_dropna_unique: setup error — {type(_e).__name__}: {_e}")

# L3 edge – empty input on matching pandas/Polars fixtures
try:
    _before_empty = before_session_dropna_unique(FIX_SESSION_DROPNA_UNIQUE_EMPTY_INFO)
    _gen_empty = gen_session_dropna_unique(FIX_SESSION_DROPNA_UNIQUE_EMPTY_INFO_PL)
    compare(_before_empty, _gen_empty, "L3 edge session_dropna_unique empty")
except Exception as _e:
    print(f"❌ L3 edge session_dropna_unique: {type(_e).__name__}: {_e}")
